# qwen-honest-DeepConf: Probe-Weighted Majority Voting for Qwen3.6-35B-A3B

Ship better Qwen3.6 for users via inference-only wrapper.

Method adapted from DeepConf (arxiv 2508.15260):
1. Generate N=8 parallel traces with temp 0.7
2. Hook L11 residual during generation
3. Apply qwen-honest probe to each trace
4. Weighted majority vote by probe confidence

Expected: +2-4pp SuperGPQA over greedy. Budget ~45-60 min.

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path
def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception: ok = False
if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'hf_transfer')
    pip('uninstall', '-y', 'transformers', 'causal-conv1d')
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC)
    pip('install','--no-cache-dir','flash-linear-attention')
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'): del sys.modules[m]

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, re, time, random, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from collections import defaultdict, Counter
from safetensors import safe_open
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from huggingface_hub import snapshot_download, HfApi, create_repo
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/deepconf'); OUT.mkdir(exist_ok=True)
HF_OUT = 'caiovicentino1/qwen36-deepconf-probe'
HF_STAGE_B = 'caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b'
MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'

PROBE_LAYER = 11
N_TRACES = 8
TEMP = 0.7
MAX_NEW = 400
N_EVAL_PROMPTS = 50

print('setup done')

## Cell 2 — Load model + L11 residual hook

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print(f'Model: {torch.cuda.memory_allocated()/1e9:.1f} GB')

def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers')]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

captured_l11 = {'chunks': []}

def l11_hook(module, inp, out):
    h = out[0] if isinstance(out, tuple) else out
    captured_l11['chunks'].append(h[:, -1, :].detach().float().cpu())

layer_L = get_layer_module(model, PROBE_LAYER)
h_handle = layer_L.register_forward_hook(l11_hook)
print(f'OK L{PROBE_LAYER} hook installed')

## Cell 3 — Train qwen-honest probe from Stage B

In [ ]:
corpus = snapshot_download(HF_STAGE_B, repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))
print(f'{len(shards)} shards')

X_list = []; y_list = []
for shard in tqdm(shards, desc='load Stage B'):
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        rs = json.loads(meta['rollouts'])
        keys = list(f.keys())
        for r in rs:
            if r.get('pred') is None: continue
            candidates = [k for k in keys if 'l11' in k.lower()]
            if not candidates: continue
            key = candidates[0]
            act = f.get_tensor(key)
            if act.dim() == 2:
                emb = act.float().mean(dim=0).numpy()
            elif act.dim() == 1:
                emb = act.float().numpy()
            elif act.dim() == 3:
                emb = act.float().reshape(-1, act.shape[-1]).mean(dim=0).numpy()
            else:
                continue
            X_list.append(emb)
            y_list.append(int(r.get('correct', False)))
            break

X = np.stack(X_list)
y = np.array(y_list)
print(f'Probe data: {X.shape}, correct rate {y.mean():.3f}')

X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
probe = LogisticRegression(max_iter=2000, C=0.5, class_weight='balanced')
probe.fit(X_tr, y_tr)
train_auc = roc_auc_score(y_tr, probe.predict_proba(X_tr)[:,1])
val_auc = roc_auc_score(y_va, probe.predict_proba(X_va)[:,1])
print(f'Probe: train AUROC={train_auc:.3f}, val AUROC={val_auc:.3f}')

with open(OUT / 'probe_l11.pkl', 'wb') as f:
    pickle.dump(probe, f)
print('OK probe saved')

## Cell 4 — PWMV wrapper functions

In [ ]:
def extract_answer(text):
    post = text.split("</think>")[-1] if "</think>" in text else text
    m = re.search(r"\\boxed\{([A-J])\}", post)
    if m: return m.group(1)
    m = re.search(r"\banswer\s*is\s*\(?([A-J])\)?", post, re.I)
    if m: return m.group(1).upper()
    m = re.findall(r"\\boxed\{([A-J])\}", text)
    if m: return m[-1]
    letters = re.findall(r"\b([A-J])\b", post[-200:] if post else text[-200:])
    return letters[-1] if letters else None

def format_prompt(q, opts):
    choices = "\n".join(f"{chr(65+i)}. {o}" for i, o in enumerate(opts))
    content = ("Answer the following multiple-choice question. "
        "Reason step by step, then put the letter in \\boxed{}.\n\n"
        f"Question: {q}\n\nOptions:\n{choices}")
    return tok.apply_chat_template([{"role":"user","content":content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)

def generate_greedy(ids):
    captured_l11["chunks"] = []
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    gen = out[0, ids.shape[1]:].tolist()
    text = tok.decode(gen, skip_special_tokens=True)
    return extract_answer(text)

def generate_parallel_traces(ids, n_traces=N_TRACES, temp=TEMP):
    captured_l11["chunks"] = []
    with torch.no_grad():
        out = model.generate(
            ids, max_new_tokens=MAX_NEW, do_sample=True, temperature=temp,
            num_return_sequences=n_traces,
            pad_token_id=tok.pad_token_id, use_cache=True, top_p=0.95)
    chunks = captured_l11["chunks"]
    if len(chunks) < 2:
        return [], []
    gen_stack = torch.stack(chunks[1:], dim=0)
    trace_embs = gen_stack.mean(dim=0).numpy()
    probe_scores = probe.predict_proba(trace_embs)[:, 1]
    answers = []
    for i in range(n_traces):
        gen = out[i, ids.shape[1]:].tolist()
        text = tok.decode(gen, skip_special_tokens=True)
        answers.append(extract_answer(text))
    return answers, probe_scores.tolist()

def weighted_vote(answers, weights):
    scores = defaultdict(float)
    for a, w in zip(answers, weights):
        if a: scores[a] += w
    if not scores: return None
    return max(scores, key=scores.get)

def naive_vote(answers):
    filtered = [a for a in answers if a]
    if not filtered: return None
    return Counter(filtered).most_common(1)[0][0]

print("PWMV functions ready")

## Cell 5 — Load held-out eval prompts

In [ ]:
questions = []
seen = set()
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q = meta['question']
        if q in seen: continue
        seen.add(q)
        opts = json.loads(meta['options'])
        if len(opts) != 10: continue
        questions.append(dict(question=q, options=opts, gold_letter=meta['gold_letter']))

random.seed(123)
random.shuffle(questions)
eval_set = questions[:N_EVAL_PROMPTS]
print(f'{len(eval_set)} eval prompts')

## Cell 6 — Benchmark 4 methods

In [ ]:
results = {'greedy': [], 'naive_vote': [], 'pwmv': [], 'probe_best_of_n': []}
t0 = time.time()
for qi, q in enumerate(tqdm(eval_set, desc='eval')):
    p = format_prompt(q['question'], q['options'])
    ids = tok(p, return_tensors='pt').input_ids.cuda()
    if ids.shape[1] > 1536: continue
    gold = q['gold_letter']
    try:
        g_ans = generate_greedy(ids)
        results['greedy'].append((g_ans, g_ans == gold if g_ans else False))
    except Exception:
        results['greedy'].append((None, False))
    try:
        answers, scores = generate_parallel_traces(ids, n_traces=N_TRACES, temp=TEMP)
        naive = naive_vote(answers)
        pwmv = weighted_vote(answers, scores)
        best_i = int(np.argmax(scores)) if scores else 0
        best = answers[best_i] if answers else None
        results['naive_vote'].append((naive, naive == gold if naive else False))
        results['pwmv'].append((pwmv, pwmv == gold if pwmv else False))
        results['probe_best_of_n'].append((best, best == gold if best else False))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        for m in ['naive_vote', 'pwmv', 'probe_best_of_n']:
            results[m].append((None, False))
    except Exception:
        for m in ['naive_vote', 'pwmv', 'probe_best_of_n']:
            results[m].append((None, False))
    if (qi+1) % 10 == 0:
        g_acc = np.mean([r[1] for r in results['greedy']])
        n_acc = np.mean([r[1] for r in results['naive_vote']])
        p_acc = np.mean([r[1] for r in results['pwmv']])
        b_acc = np.mean([r[1] for r in results['probe_best_of_n']])
        print(f'  [{qi+1}/{len(eval_set)}] greedy={g_acc:.3f}  naive={n_acc:.3f}  pwmv={p_acc:.3f}  bo{N_TRACES}={b_acc:.3f}')

h_handle.remove()

print(f'\n=== Final on {len(eval_set)} prompts, {(time.time()-t0)/60:.1f} min ===')
for name, res in results.items():
    acc = np.mean([r[1] for r in res])
    n = len([r for r in res if r[0]])
    print(f'{name:20s}: accuracy = {acc:.3f}  ({n}/{len(res)} valid)')

## Cell 7 — Upload to HF

In [ ]:
summary = {
    'model': MODEL_ID,
    'n_prompts': len(eval_set),
    'n_traces_per_prompt': N_TRACES,
    'temperature': TEMP,
    'max_new_tokens': MAX_NEW,
    'results': {name: float(np.mean([r[1] for r in res])) for name, res in results.items()},
    'probe_layer': PROBE_LAYER,
    'probe_val_auroc': float(val_auc),
}

with open(OUT / 'results.json', 'w') as f:
    json.dump({name: [(r[0], bool(r[1])) for r in res] for name, res in results.items()}, f, indent=2)
with open(OUT / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

readme = f'''---
license: apache-2.0
base_model: Qwen/Qwen3.6-35B-A3B
tags: [inference-wrapper, majority-vote, confidence-probe, reasoning]
---

# qwen-honest-DeepConf: Probe-Weighted Majority Voting

Inference wrapper for Qwen3.6-35B-A3B that combines DeepConf-style parallel traces
with qwen-honest L11 residual probe as confidence signal.

## Results (N={len(eval_set)} prompts SuperGPQA)

| method | accuracy |
|---|---|
| greedy (baseline) | {summary["results"]["greedy"]:.3f} |
| naive majority vote (N={N_TRACES}) | {summary["results"]["naive_vote"]:.3f} |
| probe best-of-N | {summary["results"]["probe_best_of_n"]:.3f} |
| PWMV (probe-weighted vote) | {summary["results"]["pwmv"]:.3f} |

## Method

Generate N={N_TRACES} traces temp {TEMP}, hook L{PROBE_LAYER} residual, mean-pool, probe-score each trace (AUROC {val_auc:.3f}), weighted majority vote.
'''
with open(OUT / 'README.md', 'w') as f: f.write(readme)

create_repo(HF_OUT, repo_type='model', private=False, exist_ok=True)
api = HfApi()
for fp in [OUT/'probe_l11.pkl', OUT/'summary.json', OUT/'results.json', OUT/'README.md']:
    try:
        api.upload_file(path_or_fileobj=str(fp), path_in_repo=fp.name,
                        repo_id=HF_OUT, repo_type='model',
                        commit_message=f'upload {fp.name}')
        print(f'OK {fp.name}')
    except Exception as e:
        print(f'FAIL {fp.name}: {e}')
print(f'\nhttps://huggingface.co/{HF_OUT}')